In [ ]:
# %%capture
# =============================================================================
# UNISIM Validation (single or suite) — Preset-driven, plug-and-play entry point
# =============================================================================
from pathlib import Path
import sys, os, warnings
from typing import List, Dict
import pandas as pd
from IPython.display import display

# Silence TF noise (optional)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("ABSL_LOG_LEVEL", "3")
warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow_addons")

# -----------------------------------------------------------------------------
# Bootstrap project root + src import path
# -----------------------------------------------------------------------------
try:
    project_root = Path(get_ipython().run_line_magic("pwd")[0].split("/notebooks")[0])
except Exception:
    project_root = Path.cwd().parent.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from forecast_pipeline.io_utils import configure_logging
configure_logging()

from common.experiment_context import ExperimentContext
from hpo.pivot_validation import (
    run_single_validation_entry,
    run_suite_validation_entry,
    resolve_experiments_for_arch,
)
from hpo.validation_suite import ValidationExperiment

# -----------------------------------------------------------------------------
# Presets (imported from src/forecast_pipeline/arps_offline.py)
# -----------------------------------------------------------------------------
from forecast_pipeline.arps_offline import (
    PIPELINE_PRESETS,
    show_pipeline_presets,
    build_pipeline_config_overrides,
)

# -----------------------------------------------------------------------------
# NEW: One-button method resolver (you added these to src/common/common.py)
# -----------------------------------------------------------------------------
from common.common import resolve_method, summarize_methods

# =============================================================================
# Control Panel (ONLY adjust here)
# =============================================================================
METHOD = "ARPS_ENSEMBLE"  # <-- ONE BUTTON
# Options:
#   "ARPS_PURE"
#   "PINN_PURE"
#   "DARTS_PURE"
#   "PINN_PLUS_ANALYTIC"
#   "ARPS_ENSEMBLE"

EXPERIMENT = "HPO_156_Lag_100_Horizon_150"
ENSEMBLE = 1

# ---- User knobs (the only ones you should edit) ----
SEED = 42
PLOT = True
LAG_WINDOW = 100
HORIZON = 150
TEST_SIZE = 0.5
VAL_SIZE = 0.15
RUN_PARAMS = {"ensemble_size": ENSEMBLE, "max_workers": 1}

# =============================================================================
# Resolve METHOD -> arch + preset + architecture_name
# =============================================================================
print(summarize_methods())
spec = resolve_method(METHOD)

arch = spec.arch
PIPELINE_PRESET = spec.pipeline_preset
architecture_name = spec.architecture_name  # used in FILTERS

# =============================================================================
# Context / run controls
# =============================================================================
CTX = ExperimentContext(group=EXPERIMENT, arch=arch)

MASTER_PROFILE_FILENAME = "final_validation_of_champions.csv"

# Make run name reflect the chosen method (avoids confusion in results dirs)
BASE_RUN_NAME = f"validation_{METHOD.lower()}"  # results/<family>/<run_name>

EXECUTION_MODE = "interactive"
CLEAR_RESULTS_BEFORE_RUN = True

RUN_MODE = "single"  # "single" | "suite"
COMPARE_WITH_HPO = True
FORCE_OVERWRITE = True  # If True, auto-confirm deletion of old CSVs in suite mode

FILTERS: Dict[str, object] = {
    "dataset": "VOLVE",
    "well": "15/9-F-14",
    "architecture_name": architecture_name,  # None for arps/darts; "Seq2PIN" for seq2
}

FILTERS: Dict[str, object] = {
    "dataset": "UNISIM_IV",
    "well": "P12",
    "architecture_name": architecture_name,
}

# =============================================================================
# Preset catalog (brief explanation for the user)
# =============================================================================
show_pipeline_presets()

if PIPELINE_PRESET not in PIPELINE_PRESETS:
    raise ValueError(
        f"Unknown PIPELINE_PRESET={PIPELINE_PRESET!r} resolved from METHOD={METHOD!r}. "
        f"Choose one of: {sorted(PIPELINE_PRESETS)}"
    )

print(f"\nSelected METHOD: {METHOD}")
print(f"Resolved arch: {arch}")
print(f"Resolved preset: {PIPELINE_PRESET}")
print(f"Meaning: {PIPELINE_PRESETS[PIPELINE_PRESET].description}\n")

# =============================================================================
# CONFIG_OVERRIDES (built from preset + minimal user knobs)
# =============================================================================
JOB_KNOBS = {
    "seed": SEED,
    "plot": PLOT,
    "lag_window": LAG_WINDOW,
    "horizon": HORIZON,
    "test_size": TEST_SIZE,
    "val_size": VAL_SIZE,
}

CONFIG_OVERRIDES = build_pipeline_config_overrides(
    preset=PIPELINE_PRESET,
    job_knobs=JOB_KNOBS,
    run_params=RUN_PARAMS,
)

# Make sure outputs go to the current experiment folder
CONFIG_OVERRIDES.setdefault("infra", {})["experiments_output_dir"] = str(CTX.results_dir)

# Grid for RUN_MODE="suite" (will be overridden to [reconstruct] if arch="darts")
EXPERIMENTS: List[ValidationExperiment] = [
    ValidationExperiment(policy="reconstruct"),
    ValidationExperiment(policy="hp_hist"),
    ValidationExperiment(policy="reconstruct_warm_raw"),
    ValidationExperiment(policy="reconstruct_warm_hp"),
    ValidationExperiment(policy="reconstruct_warm_ewma"),
    ValidationExperiment(policy="reconstruct_warm_holt"),
]

# =============================================================================
# Main — notebook entry point
# =============================================================================
def main() -> None:
    # Enforce arch-specific experiment policy
    resolved_experiments = resolve_experiments_for_arch(arch, EXPERIMENTS)

    if RUN_MODE == "single":
        _ = run_single_validation_entry(
            project_root=project_root,
            ctx=CTX,
            master_profile_filename=MASTER_PROFILE_FILENAME,
            run_name=BASE_RUN_NAME,
            filters=FILTERS,
            execution_mode=EXECUTION_MODE,
            ensemble_size=ENSEMBLE,
            config_overrides=CONFIG_OVERRIDES,
            delete_previous_results=CLEAR_RESULTS_BEFORE_RUN,
            compare_with_hpo=COMPARE_WITH_HPO,
        )
    elif RUN_MODE == "suite":
        _ = run_suite_validation_entry(
            project_root=project_root,
            ctx=CTX,
            base_run_name=BASE_RUN_NAME,
            master_profile_filename=MASTER_PROFILE_FILENAME,
            filters=FILTERS,
            template_overrides=CONFIG_OVERRIDES,
            experiments=resolved_experiments,
            execution_mode=EXECUTION_MODE,
            ensemble_size=ENSEMBLE,
            force_overwrite=FORCE_OVERWRITE,
        )
    else:
        raise ValueError("RUN_MODE must be either 'single' or 'suite'.")

if __name__ == "__main__":
    main()
